# Text classification - movie reviews on imdb and rotten_tomatoes dataset

## Option 1 – Encoder-based models: Using a Task-Specific Model (task: classification)

In this section, we will use a **pretrained encoder-based transformer model** to perform **sentiment classification** on a dataset of movie reviews.

We’ll start by **loading the Rotten Tomatoes dataset** using hugging_face `datasets.load_dataset`.

Each example in the dataset contains:
- `'text'`: the review (a string)
- `'label'`: the sentiment label (0 = negative, 1 = positive)

We can explore the dataset by printing specific examples.  
Here, we use **fancy indexing** to fetch both the **first and last example** in the training split:

```python
data['train'][0, -1]


In [ ]:
# Data load
from datasets import load_dataset
data = load_dataset('rotten_tomatoes')
# fancy indexing, gives both the first and last rows in a single call
data['train'][0,-1]

### Memory Clean-up
Before loading a new model, we ensure memory is cleared (especially useful when using Apple's Metal backend).

In [446]:

def cleanup_mps_memory():
    """
    Frees MPS memory by deleting global variables 'model' and 'tokenizer' if they exist.
    Useful when you want to avoid passing model/tokenizer manually.
    """
    import gc
    import torch

    for var in ['model', 'tokenizer']:
        if var in globals():
            print(f"🔹 Deleting: {var}")
            del globals()[var]

    gc.collect()
    torch.mps.empty_cache()
    print("MPS memory cleaned.")
cleanup_mps_memory()

🔹 Deleting: model
🔹 Deleting: tokenizer
MPS memory cleaned.


In [488]:
# Suppress transformers + general warnings when loading models/pipelines
import warnings
from transformers import logging

# Suppress all warnings
warnings.filterwarnings("ignore")

# Suppress transformers warnings
logging.set_verbosity_error()

### Loading a pretrained transformer model for sentiment classification

Next, we load a **RoBERTa-based transformer model fine-tuned for sentiment analysis**.

We’ll use **`transformers.pipeline`** to simplify inference:
- Automatically loads the model + tokenizer
- Handles tokenization, preprocessing, model forward pass
- Supports **returning scores for all possible labels** (`return_all_scores=True`)

We also select the **best available hardware** for inference:
1. GPU (`cuda`)
2. Apple Silicon (`mps`)
3. CPU (fallback)

We set the `device` dynamically, and pass it to the pipeline.

This model (`cardiffnlp/twitter-roberta-base-sentiment-latest`) was trained on Twitter sentiment data, but works well on short reviews.
We also enable `truncation=True` to safely handle long input texts.


In [451]:
from transformers import pipeline
import torch 

device = torch.device('cuda' if torch.cuda.is_available() else 
                     'mps' if torch.backends.mps.is_available() else 'cpu')

model_path = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
pipe = pipeline(
    model = model_path,
    tokenizer=model_path,
    return_all_scores=True,
    device=device,
    truncation=True
)

### Preparing the test dataset for inference

We now prepare the **test data** to pass into the pipeline.

Key steps:
- We import `KeyDataset`, a utility that allows us to **select a single column from a dataset**.
- Instead of manually looping through rows, `KeyDataset` extracts just the **`'text'` column** from our dataset (efficient + cleaner).
-  Useful when passing input directly into a pipeline.

Let’s print:
- The `KeyDataset` object itself
- The **first example** inside it (to confirm it returns plain text strings)

This prepares our test set for looping through predictions!


In [457]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

print(KeyDataset(data['test'], 'text'))
print('first example: ', KeyDataset(data['test'], 'text')[0])

first example:  lovingly photographed in the manner of a golden book sprung to life , stuart little 2 manages sweetness largely without stickiness .


### Getting a prediction for a single example

Let’s test our pipeline on **just the first example** to see the raw output.

We pass one input (the first text in our test set) to `pipe()` to get its **predicted class scores**.

The output will be a list of dictionaries with:
- `label`: class name (e.g., negative, neutral, positive)
- `score`: probability/confidence for that class

This helps us understand the pipeline’s output format before running on the full dataset.

As we can see, the model predicted this review as positive (95% probability)

In [458]:
pipe(KeyDataset(data['test'], 'text')[0])

[[{'label': 'negative', 'score': 0.00516123790293932},
  {'label': 'neutral', 'score': 0.04023353382945061},
  {'label': 'positive', 'score': 0.9546052813529968}]]

### Running inference on the **entire test dataset**

Now we process **all examples** in the test set to predict their sentiment.

**Key points:**
- We loop over each output from the pipeline
- For each output:
  - Extract the `negative` and `positive` scores
  - Ignore the `neutral` score (index 1)
  - Choose the label with the highest score using `np.argmax`
- Store the predicted label in `y_pred`

We use `tqdm` to track progress visually.

At the end, `y_pred` will contain one predicted label (0 for negative, 1 for positive) per input in the test set.


In [423]:
y_pred = []
for output in tqdm(pipe(KeyDataset(data['test'], 'text')), total=len(data['test'])):
    negative_score = output[0]['score']
    positive_score = output[2]['score']
    assignment = np.argmax([negative_score, positive_score])
    y_pred.append(assignment)

100%|███████████████████████████████████████| 1066/1066 [00:12<00:00, 85.06it/s]


### Evaluating model performance

Now that we have predictions for the test set, we can evaluate how well the model performed.

We use `sklearn`'s `classification_report` to calculate key metrics:
- **Precision**: How many predicted positives are truly positive
- **Recall**: How many actual positives were correctly identified
- **F1-score**: Harmonic mean of precision and recall
- **Support**: Number of samples per class

We define a helper function `evaluate_performance` to print the report.

The evaluation compares:
- `y_true`: The ground truth labels from the dataset
- `y_pred`: Our model's predicted labels

In [424]:
from sklearn.metrics import classification_report

In [425]:
def evaluate_performance(y_true, y_pred):
    """Create and print classification report"""
    performance = classification_report(
        y_true, y_pred,
        target_names=["Negative review", "Positive Review"]
    )
    print(performance)

In [426]:
evaluate_performance(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative review       0.76      0.88      0.81       533
Positive Review       0.86      0.72      0.78       533

       accuracy                           0.80      1066
      macro avg       0.81      0.80      0.80      1066
   weighted avg       0.81      0.80      0.80      1066



### Trying a different model: `siebert/sentiment-roberta-large-english`

In this step, we swap out the earlier model for a different one:  
**`siebert/sentiment-roberta-large-english`** — a large RoBERTa model fine-tuned specifically for sentiment analysis.

We repeat the same inference and evaluation process:
1. Load the model into a `transformers` pipeline
2. Run predictions on the test dataset
3. Compute precision, recall, and F1-score

**Why try different models?**
- To compare accuracy and speed across different architectures
- Some models may generalize better depending on the dataset


In [433]:
model_path = 'siebert/sentiment-roberta-large-english'
pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    device=device,
    truncation=True
)

y_pred = []
for output in tqdm(pipe(KeyDataset(data['test'], 'text')), total=len(data['test'])):
    negative_score = output[0]['score']
    positive_score = output[1]['score']
    assignment = np.argmax([negative_score, positive_score])
    y_pred.append(assignment)

evaluate_performance(data['test']['label'], y_pred)

100%|███████████████████████████████████████| 1066/1066 [00:21<00:00, 48.46it/s]

                 precision    recall  f1-score   support

Negative review       0.93      0.91      0.92       533
Positive Review       0.91      0.93      0.92       533

       accuracy                           0.92      1066
      macro avg       0.92      0.92      0.92      1066
   weighted avg       0.92      0.92      0.92      1066



### Trying another model: `distilbert-base-uncased-finetuned-sst-2-english`

In this step, we experiment with a third model for sentiment classification:

 **Model:** `distilbert-base-uncased-finetuned-sst-2-english`  
This is a distilled version of BERT (smaller & faster) fine-tuned on SST-2 (Stanford Sentiment Treebank).

We repeat the same process:
1. Load the model with `transformers.pipeline`
2. Run predictions on the Rotten Tomatoes test dataset
3. Evaluate using precision, recall, F1-score

Why try DistilBERT?
- Faster inference (fewer parameters)
- Smaller memory footprint
- Good tradeoff between speed and accuracy

We'll compare its results to the earlier RoBERTa models later.

In [438]:
model_path = 'distilbert-base-uncased-finetuned-sst-2-english'
pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    device=device,
    truncation=True
)

y_pred = []
for output in tqdm(pipe(KeyDataset(data['test'], 'text')), total=len(data['test'])):
    negative_score = output[0]['score']
    positive_score = output[1]['score']
    assignment = np.argmax([negative_score, positive_score])
    y_pred.append(assignment)

evaluate_performance(data['test']['label'], y_pred)

100%|██████████████████████████████████████| 1066/1066 [00:09<00:00, 109.68it/s]

                 precision    recall  f1-score   support

Negative review       0.89      0.90      0.90       533
Positive Review       0.90      0.89      0.90       533

       accuracy                           0.90      1066
      macro avg       0.90      0.90      0.90      1066
   weighted avg       0.90      0.90      0.90      1066



## Results Comparison: Model Performance

Let's compare the accuracy, precision, recall, and F1-score for the three different models we tested on the Rotten Tomatoes test dataset:

| Model                                      | Accuracy | Negative F1 | Positive F1 |
|-------------------------------------------|----------|-------------|-------------|
| cardiffnlp/twitter-roberta-base-sentiment-latest | 80%      | 0.81        | 0.78        |
| siebert/sentiment-roberta-large-english    | 92%      | 0.92        | 0.92        |
| distilbert-base-uncased-finetuned-sst-2-english | 90%      | 0.90        | 0.90        |

### Observations:
- **siebert/sentiment-roberta-large-english** achieved the highest overall accuracy and balanced F1-scores.
- **distilbert-base-uncased-finetuned-sst-2-english** performed slightly lower but still very strong, with faster inference time expected due to its smaller size.
- **cardiffnlp/twitter-roberta-base-sentiment-latest** underperformed on this dataset, possibly because it was fine-tuned on Twitter sentiment data rather than movie reviews.

**Takeaway:** Model choice should depend on task domain and balance between accuracy vs. inference speed.

### Exploring a Larger Dataset: IMDB Movie Reviews

After testing on the smaller Rotten Tomatoes dataset (∼1k test samples), we now switch to the **IMDB movie reviews dataset**, which is much larger:

- **25,000 training examples**
- **25,000 test examples**
- **50,000 unsupervised unlabeled examples**

Each sample contains:
- `text`: the review text
- `label`: 0 for negative, 1 for positive

This dataset is a classic benchmark for binary sentiment classification on longer, more complex reviews compared to shorter Rotten Tomatoes snippets.

We load it using `datasets.load_dataset('imdb')`, giving us a convenient `DatasetDict` object with splits.


In [460]:
from datasets import load_dataset
data = load_dataset('imdb')
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

### Running `siebert/sentiment-roberta-large-english` on IMDB Dataset

We now apply the **`siebert/sentiment-roberta-large-english` model** to the **IMDB test dataset**.

This is a **large RoBERTa-based model fine-tuned for sentiment classification**, trained on diverse data sources for robust generalization.

We follow the same inference steps as before:

1. Create a `pipeline` for sentiment analysis.
2. Pass the IMDB test data using `KeyDataset` (so the pipeline only reads the `'text'` column).
3. For each example, extract:
   - `negative_score` → probability for class 0
   - `positive_score` → probability for class 1
4. Predict the label with `np.argmax`.
5. Evaluate using **precision, recall, F1, accuracy**.

Note: Since IMDB is 25x larger than Rotten Tomatoes, inference will take longer!


In [441]:
model_path = 'siebert/sentiment-roberta-large-english'
pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    device=device,
    truncation=True
)

y_pred = []
for output in tqdm(pipe(KeyDataset(data['test'], 'text')), total=len(data['test'])):
    negative_score = output[0]['score']
    positive_score = output[1]['score']
    assignment = np.argmax([negative_score, positive_score])
    y_pred.append(assignment)

evaluate_performance(data['test']['label'], y_pred)

100%|███████████████████████████████████| 25000/25000 [1:03:29<00:00,  6.56it/s]

                 precision    recall  f1-score   support

Negative review       0.96      0.95      0.96     12500
Positive Review       0.95      0.96      0.96     12500

       accuracy                           0.96     25000
      macro avg       0.96      0.96      0.96     25000
   weighted avg       0.96      0.96      0.96     25000



### Running `distilbert-base-uncased-finetuned-sst-2-english` on IMDB Dataset

Next, we apply the **`distilbert-base-uncased-finetuned-sst-2-english` model** to the **IMDB test dataset**.

We follow the same steps as before.

**Why DistilBERT?**
- It’s **smaller & faster** than RoBERTa while retaining ~95% of its accuracy.
- Useful for large datasets or when compute is limited.

Since IMDB has **25,000 samples**, inference will still take noticeable time — though likely faster than RoBERTa.

In [442]:
model_path = 'distilbert-base-uncased-finetuned-sst-2-english'
pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    device=device,
    truncation=True
)

y_pred = []
for output in tqdm(pipe(KeyDataset(data['test'], 'text')), total=len(data['test'])):
    negative_score = output[0]['score']
    positive_score = output[1]['score']
    assignment = np.argmax([negative_score, positive_score])
    y_pred.append(assignment)

evaluate_performance(data['test']['label'], y_pred)

100%|█████████████████████████████████████| 25000/25000 [11:02<00:00, 37.76it/s]

                 precision    recall  f1-score   support

Negative review       0.87      0.92      0.89     12500
Positive Review       0.91      0.86      0.89     12500

       accuracy                           0.89     25000
      macro avg       0.89      0.89      0.89     25000
   weighted avg       0.89      0.89      0.89     25000



### Results Comparison: `siebert/sentiment-roberta-large-english` vs `distilbert-base-uncased-finetuned-sst-2-english`

Let's compare the performance of the two models on the **IMDB test set (25,000 samples)**:

| Model                                    | Accuracy | F1-Score (Neg) | F1-Score (Pos) | Inference Speed  |
|-----------------------------------------|----------|----------------|----------------|-----------------|
| siebert/sentiment-roberta-large-english  | 0.96     | 0.96           | 0.96           | ~63 min (6.56 it/s) |
| distilbert-base-uncased-finetuned-sst-2-english | 0.89     | 0.89           | 0.89           | ~11 min (37.76 it/s) |

#### Observations:
- **siebert/sentiment-roberta-large-english** achieved **higher accuracy and F1-scores** across both classes.
- **distilbert-base-uncased-finetuned-sst-2-english** is usually **~6 times faster** in inference but had slightly lower accuracy. (uou check the inference time with time package)
- Tradeoff: **better performance vs faster inference** → choice depends on application needs (speed vs accuracy).

#### Why such differences?
- RoBERTa-large (used in `siebert/...`) is a **much larger model** → more parameters, better contextual understanding → slower inference.
- DistilBERT is a **compressed/smaller model** → fewer parameters → faster but a bit less accurate.

## Option 2 - Encoder-based models: using a model that generates embeddings (task: embedding) + supervised ML (if we have labels)

In this approach, instead of directly using a pre-trained model to classify text, we use the model to **convert text into numerical vector representations (embeddings)**.

These embeddings can then be used as input to a **traditional machine learning classifier** (like logistic regression, SVM, etc.) to perform classification.

In this example:
- We use the pre-trained model `sentence-transformers/all-mpnet-base-v2` to generate 768-dimensional embeddings for each movie review.
- We will train a **logistic regression classifier** on these embeddings to predict positive/negative sentiment.

This approach is useful when:
- You want more control over the classification model.
- You want to reuse the embeddings for multiple downstream tasks.

Let’s generate the embeddings for the **IMDB dataset**.


In [461]:
from sentence_transformers import SentenceTransformer

In [462]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
train_embeddings = model.encode(data['train']['text'], show_progress_bar=True)
test_embeddings = model.encode(data['test']['text'], show_progress_bar=True)

Batches: 100%|████████████████████████████████| 782/782 [30:19<00:00,  2.33s/it]


The shape of `train_embeddings` is `(25000, 768)`.

- **25000** → number of samples in the IMDB training dataset
- **768** → the embedding dimensionality of the `all-mpnet-base-v2` model

Each movie review has been encoded into a 768-dimensional dense vector. These embeddings capture semantic information from the text, ready to be used as input for downstream machine learning models like logistic regression.

In [463]:
train_embeddings.shape

(25000, 768)

### Now we train a simple **logistic regression classifier** using the embeddings as features:

- `train_embeddings`: the 768-dimensional vectors we generated for each review
- `data['train']['label']`: the true sentiment labels (0 = negative, 1 = positive)

Logistic regression will learn a linear decision boundary in this 768-dimensional space to separate positive from negative reviews.

This step is **supervised learning** on top of precomputed embeddings (instead of fine-tuning the transformer model itself).

In [464]:
from sklearn.linear_model import LogisticRegression

In [465]:
clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data['train']['label'])

LogisticRegression(random_state=42)

We use the trained logistic regression model to predict sentiment labels on the test set.

Then we evaluate the performance with `classification_report`:
**Overall accuracy: 89%**

This shows that even without fine-tuning the transformer, just using **frozen sentence embeddings + logistic regression** achieves strong results on IMDb sentiment classification.

This method is **faster and cheaper** than full fine-tuning but slightly less powerful than task-specific fine-tuned models.

In [466]:
y_pred = clf.predict(test_embeddings)
evaluate_performance(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative review       0.90      0.88      0.89     12500
Positive Review       0.89      0.90      0.89     12500

       accuracy                           0.89     25000
      macro avg       0.89      0.89      0.89     25000
   weighted avg       0.89      0.89      0.89     25000



## Option 3 - Encoder-based models: using a model that generates embeddings (task: embedding) + Unsupervised Zero-shot classification

In this approach, we don't train a classifier.  
We simply:

1. Encode each test sentence into an embedding.
2. Encode **label descriptions** ("A negative review" and "A positive review") into embeddings.
3. Use **cosine similarity** to measure how close each test sentence is to the label descriptions.
4. Assign the label with the highest similarity.

This is an example of **zero-shot classification using embedding similarity**.

---

#### Results:

| Metric        | Negative | Positive |
|---------------|----------|----------|
| Precision      | 0.65     | 0.91     |
| Recall         | 0.95     | 0.48     |
| F1-score       | 0.77     | 0.63     |

**Accuracy: 72%**

---

We see this approach achieves **moderate performance** without any supervised training.  
It **predicts negative reviews well (high recall)** but **struggles with positive reviews (lower recall)**.

This happens because the embedding space isn't explicitly optimized for this task; it's relying only on general semantic similarity.

This zero-shot method can be useful for **quick insights or when no labeled data is available** but isn't as strong as supervised models.

In [467]:
from sklearn.metrics.pairwise import cosine_similarity

In [468]:
label_embeddings = model.encode(['A negative review', 'A positive review'])
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)
evaluate_performance(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative review       0.65      0.95      0.77     12500
Positive Review       0.91      0.48      0.63     12500

       accuracy                           0.72     25000
      macro avg       0.78      0.72      0.70     25000
   weighted avg       0.78      0.72      0.70     25000



### Testing different label descriptions in Zero-shot classification

We tried **modifying the label descriptions** from:

- `"A negative review"` → `"A very negative movie review"`
- `"A positive review"` → `"A very positive movie review"`

Why?  
Because adding more **domain-specific and descriptive phrases** can sometimes improve similarity-based zero-shot performance.

We re-encoded these new labels and re-ran cosine similarity classification.

---

#### Results:

| Metric        | Negative | Positive |
|---------------|----------|----------|
| Precision      | 0.75     | 0.92     |
| Recall         | 0.94     | 0.69     |
| F1-score       | 0.83     | 0.79     |

**Accuracy: 81%**

---

**Accuracy improved from 72% → 81%** compared to using the original simple labels!

The model still predicts **negative reviews very well (high recall)** and improves positive review precision.

This shows how **label wording in zero-shot embedding-based classification can influence results** — crafting more task-relevant, detailed labels can help the model "understand" the task better without retraining.

**Key takeaway:**  
Zero-shot embedding-based classification is sensitive to label phrasing → always experiment with label wording!

In [469]:
label_embeddings = model.encode(['A very negative movie review', 'A very positive movie review'])
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)
evaluate_performance(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative review       0.75      0.94      0.83     12500
Positive Review       0.92      0.69      0.79     12500

       accuracy                           0.81     25000
      macro avg       0.83      0.81      0.81     25000
   weighted avg       0.83      0.81      0.81     25000



## Option 4 – Text classification using generative (decoder-based) models

In this section, we experiment with **decoder-based generative models** for text classification.

We use the **T5 model** (specifically `google/flan-t5-small`) with a 🤗 `pipeline` for **text-to-text generation**:

#### About T5
* **T5 (Text-to-Text Transfer Transformer)** was introduced by Google Research.
* It frames **every NLP task as a text-to-text task**: both input and output are treated as strings.
* Example:
   * Input: `"translate English to French: That is good"`
   * Output: `"C'est bon"`
* This unified format enables T5 to tackle translation, summarization, classification, and more using the same architecture.
* Under the hood, T5 is an **encoder-decoder Transformer**, trained using a **denoising objective** similar to masked language modeling.

#### About FLAN-T5
* **FLAN-T5** is an improved version of T5, trained with **instruction tuning**.
* Instruction tuning exposes the model to **human-written task instructions** (e.g., "Summarize this paragraph", "Classify the sentiment").
* This process makes FLAN-T5 **better at following prompts and instructions without needing task-specific fine-tuning**.
* Compared to the original T5, FLAN-T5 achieves **stronger zero-shot and few-shot performance** across many benchmarks.
* For our use case, this means FLAN-T5 can answer prompts like "Is the following review positive or negative?" simply from the prompt wording.

In [470]:
pipe = pipeline(
    'text2text-generation',
    model='google/flan-t5-small',
    device=device
)

### Preparing input prompts for T5

T5 expects every task to be phrased as **a text-to-text instruction**.

We create a prompt like:

> *“Is the following sentence positive or negative?”*

and prepend it to every input review.

We use `Dataset.map` to apply this transformation to all rows in the dataset, adding a new column called `'t5'`

In [471]:
prompt = 'Is the following sentence positive or negative? '
data = data.map(lambda example: {'t5': prompt + example['text']})

Map: 100%|██████████████████████| 50000/50000 [00:00<00:00, 74542.32 examples/s]


In [472]:
data['train']['t5'][0]

'Is the following sentence positive or negative? I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the s

### Running inference with T5 on the IMDB dataset

We use the `pipeline` with **T5** to generate a textual answer (`"positive"` or `"negative"`).  
We loop through the test set and collect predictions.

The generated output is mapped:

- `"negative"` → `0`
- `"positive"` → `1`

Then we evaluate performance with classification report. 

In [473]:
y_pred = []
for output in tqdm(pipe(KeyDataset(data['test'], 't5')), total=len(data['test'])):
    text = output[0]['generated_text']
    y_pred.append(0 if text == 'negative' else 1)

100%|██████████████████████████████████| 25000/25000 [16:47:03<00:00,  2.42s/it]


In [474]:
evaluate_performance(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative review       0.93      0.92      0.92     12500
Positive Review       0.92      0.93      0.92     12500

       accuracy                           0.92     25000
      macro avg       0.92      0.92      0.92     25000
   weighted avg       0.92      0.92      0.92     25000



### Experiment: Changing the prompt for T5

We try a slightly different prompt wording:  
**"Is the following movie review positive or negative?"** (instead of "sentence")

We also **store both the raw generated text (`text_pred`) and the numeric prediction (`y_pred`)**.

Interesting:
- The accuracy stays at 92%, similar to the earlier prompt.
- However, some individual predictions might shift depending on phrasing.

We could further explore:
- Which samples flipped labels?
- Does prompt wording affect calibration or confidence?

In [475]:
prompt = 'Is the following movie review positive or negative? '
data = data.map(lambda example: {'t5': prompt + example['text']})
text_pred = []
y_pred = []
for output in tqdm(pipe(KeyDataset(data['test'], 't5')), total=len(data['test'])):
    text = output[0]['generated_text']
    text_pred.append(text)
    y_pred.append(0 if text == 'negative' else 1)
evaluate_performance(data['test']['label'], y_pred)

100%|█████████████████████████████████████| 25000/25000 [44:12<00:00,  9.43it/s]


                 precision    recall  f1-score   support

Negative review       0.91      0.93      0.92     12500
Positive Review       0.93      0.91      0.92     12500

       accuracy                           0.92     25000
      macro avg       0.92      0.92      0.92     25000
   weighted avg       0.92      0.92      0.92     25000



### Using **GPT** for classification (via OpenAI API)

In this section, we use **GPT-3.5-turbo** (hosted via OpenAI API) to classify text.

We manually craft a prompt to query the model in a **zero-shot classification** setting.

Key notes:
- This is a zero-shot inference (no fine-tuning).
- The prompt controls the behavior; careful wording is important!
- Using temperature=0 makes output deterministic.

First, create a .env file with OpenAI API and load it into your environment:

In [476]:
# with open(".env", "w") as f:
#     f.write("OPENAI_API_KEY=YOUR_KEY_GOES_HERE\n")

In [477]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [478]:
import openai

In [479]:
client = openai.OpenAI(api_key=api_key)

We create a function `gpt_generation` to handle:
- Setting up the system & user messages
- Sending them to the OpenAI API
- Returning the generated response

In [480]:
def gpt_generation(prompt, document, model='gpt-3.5-turbo-0125'):
    messages=[
        {
            'role': 'system',
            'content': 'you are helpful assistant'
        },
        {
            'role': 'user',
            'content': prompt.replace("[DOCUMENT]", document)
        }
    ]
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=model,
        temperature=0
    )
    return chat_completion.choices[0].message.content

We define a prompt asking GPT to classify sentiment:

In [481]:
prompt = """Predict whether the following document is a positive or negative movie review:

[DOCUMENT]

If it is positive return 1 and if it is negative return 0. Do not give any other answers.
"""

In [482]:
document = 'what a funny movie'
gpt_generation(prompt, document)

'1'